In [16]:
import numpy as np
import pandas as pd
import re
from sklearn.neighbors import BallTree

## Inspect and clean FAULTS dataset

In [17]:
faults_df = pd.read_csv(filepath_or_buffer="../data/J1939Faults.csv", low_memory=False)

In [18]:
faults_df.sample(5)

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
604467,619911,12432636,2016-10-27 12:45:42.000,Low (Severity High) Wheel Sensor ABS Axle 1 Left,NaN,BB41103* BB41104*,S121200083,EC60-adv,BNDWS,11,789,1,True,127,NaN,1557,105338698,41.073009,-87.866805,2016-10-27 12:46:18.000
910993,936805,35622722,2017-12-27 15:57:27.000,Low (Severity Medium) Engine Coolant Level,NaN,05317106*04346433*050515205406*09400034*G1*BDR*,79844880,6X1u13D1500000000,CMMNS,0,111,18,True,3,NaN,1805,105407611,34.989722,-85.471296,2017-12-27 15:58:03.000
166077,168463,4081193,2015-08-16 11:49:09.000,High Voltage (Left Fuel Level Sensor),NaN,unknown,unknown,unknown,unknown,49,829,3,True,126,NaN,1692,105320363,35.996574,-83.713009,2015-08-16 11:49:05.000
302745,307251,6248149,2015-12-10 12:06:16.000,Low (Severity Low) Engine Coolant Level,NaN,04993120*00176496*082113134117*07700053*I0*BBZ*,79621364,6X1u10D1500000000,CMMNS,0,111,17,False,1,NaN,1597,105344243,35.544814,-86.182592,2015-12-10 12:06:11.000
1072405,1116158,76487530,2019-02-06 07:05:31.000,High Voltage (Fuel Level),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,96,3,False,126,NaN,2000,105369490,36.195324,-83.175416,2019-02-06 07:05:27.000


In [19]:
faults_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1187335 entries, 0 to 1187334
Data columns (total 20 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   RecordID               1187335 non-null  int64  
 1   ESS_Id                 1187335 non-null  int64  
 2   EventTimeStamp         1187335 non-null  object 
 3   eventDescription       1126490 non-null  object 
 4   actionDescription      0 non-null        float64
 5   ecuSoftwareVersion     891285 non-null   object 
 6   ecuSerialNumber        844318 non-null   object 
 7   ecuModel               1122577 non-null  object 
 8   ecuMake                1122577 non-null  object 
 9   ecuSource              1187335 non-null  int64  
 10  spn                    1187335 non-null  int64  
 11  fmi                    1187335 non-null  int64  
 12  active                 1187335 non-null  bool   
 13  activeTransitionCount  1187335 non-null  int64  
 14  faultValue        

### Possible Features:
* RecordID is unique
* ESS_Id all nan
* EventTimeStamp has 1,050,909 unique values (no nan)
* eventDescription has 60,845 nan (Create 'severity' column based on descriptions)
* actionDescription all nan
* ecuSoftwareVersion has 1,899 unique values (296,050 nan)
* ecuSerialNumber all nan
* ecuModel has 30 unique values (64,758 nan)
* ecuMake has 23 unique values (64,758 nan)
* ecuSource has 5 unique values (no nan)
* spn has 450 unique values (no nan)
* fmi has 26 unique values (no nan)
* active has 2 unique values (no nan)
* activeTransitionCount has 8,128 unique values (no nan)
* faultValue all nan
* EquipmentID has 1,927 unique values (no nan)
* MCTNumber has 768 unique values (no nan)
* Latitude has 211,823 unique values (no nan)
* Longitude has 265,211 unique values (no nan)
* LocationTimeStamp has 1,036,006 unique values (no nan)

In [20]:
# Convert timestamps to datetime objects
faults_df['EventTimeStamp'] = pd.to_datetime(faults_df['EventTimeStamp'])
faults_df['LocationTimeStamp'] = pd.to_datetime(faults_df['LocationTimeStamp'])
print(f'EventTimeStamp datatype: {faults_df['LocationTimeStamp'].dtype}')

EventTimeStamp datatype: datetime64[ns]


In [21]:
# Convert SPN and FMI values to strings
faults_df['spn'] = faults_df['spn'].astype(str)
faults_df['fmi'] = faults_df['fmi'].astype(str)
print(f'spn datatype: {faults_df['spn'].dtype}')

spn datatype: object


In [22]:
# Create column for active codes that are near service stations
service_stations = [
    (36.0666667, -86.4347222),
    (35.5883333, -86.4438888),
    (36.1950, -83.174722)
]

earth_radius_km = 6371.0
    
station_radians = np.radians(service_stations)
points_radians = np.radians(faults_df[['Latitude', 'Longitude']].values)
    
tree = BallTree(station_radians, metric='haversine')
    
# Query radius in radians
indices = tree.query_radius(X=points_radians, r=1.0 / earth_radius_km)
    
faults_df['NearServiceStation'] = np.array([len(idx) > 0 for idx in indices])
faults_df['NearServiceStation'].value_counts()

NearServiceStation
False    1055968
True      131367
Name: count, dtype: int64

In [23]:
# Create column for full derate flag
# Full derate is determined by spn code 5246 and engine code active is true
faults_df['IsFullDerate'] = (
        (faults_df['spn'] == '5246')
        & faults_df['active']
        & ~faults_df['NearServiceStation']
    )
faults_df['IsFullDerate'].isna().sum()

np.int64(0)

In [24]:
# Create columns for severity level from eventDescription
def extract_severity(text):

    if pd.isna(text):
        return np.nan

    # Severity with "Low", "Medium", or "high"
    pattern = r'Severity\s+(Low|Medium|High)'

    # Find pattern
    match = re.search(pattern, text)

    if match: 
        return f"Severity {match.group(1)}"
    else: 
        return np.nan

faults_df['Severity_Level'] = faults_df['eventDescription'].apply(extract_severity)

severity_map = {
    'Severity Low': 1,
    'Severity Medium': 2,
    'Severity High': 3
}

faults_df['Severity_Level_Numeric'] = faults_df['Severity_Level'].map(severity_map)

In [25]:
# Create target columns for full derate windows
def create_target_column(window_max:float, window_min:float, df:'DataFrame'=faults_df) -> None:

    df = df.sort_values(['EquipmentID', 'EventTimeStamp'])

    column_name = f'Derate_Target'
    
    # Initialize target column
    faults_df[column_name] = 0

    # Dataframe with just the derate events
    derate_events_df = faults_df[faults_df['IsFullDerate']].copy()

    # Group by EquipmentID 
    for equipment_id, group in faults_df.groupby('EquipmentID'):
        # Get derate events for this truck only
        truck_derates = derate_events_df[derate_events_df['EquipmentID'] == equipment_id]
        
        if len(truck_derates) > 0:
            # Get indices and timestamps for this truck's rows
            truck_indices = group.index
            truck_timestamps = group['EventTimeStamp'].values
            
            # For each derate event in this truck
            for _, derate_row in truck_derates.iterrows():
                derate_time = derate_row['EventTimeStamp']
                
                # Window: 
                window_start = derate_time - pd.Timedelta(hours=window_max)
                window_end = derate_time - pd.Timedelta(hours=window_min)

                # All events in the prediction window
                in_window = (truck_timestamps >= window_start) & (truck_timestamps <= window_end)
                target_indices_to_mark = truck_indices[in_window]

                # All events within and including the derate
                imminent_derate = (truck_timestamps > window_end) & (truck_timestamps <= derate_time)
                derate_indices_to_mark = truck_indices[imminent_derate]
                
                # Marked as predicting a derate
                faults_df.loc[target_indices_to_mark, column_name] = 1
                faults_df.loc[derate_indices_to_mark, column_name] = 2                

    print(f"Total events: {len(faults_df)}")
    print(f"Other: {(faults_df[column_name] == 0).sum()}")
    print(f"Events 2-8 hrs prior to derate: {(faults_df[column_name] == 1).sum()}")
    print(f"Events 2 hrs prior to and including derate: {(faults_df[column_name] == 2).sum()}")

In [26]:
create_target_column(window_max=8.0, window_min=2.0)

Total events: 1187335
Other: 1185150
Events 2-8 hrs prior to derate: 1085
Events 2 hrs prior to and including derate: 1100


In [27]:
faults_df['Derate_Target'].isna().sum()

np.int64(0)

## Inspect DIAGNOSTICS

In [28]:
diagnostics_df = pd.read_csv(filepath_or_buffer="../data/VehicleDiagnosticOnboardData.csv", low_memory=False)
diagnostics_df

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1
...,...,...,...,...
12821621,12864020,EngineCoolantTemperature,181.4,1248457
12821622,12864021,ParkingBrake,False,1248457
12821623,12864022,SwitchedBatteryVoltage,14.1,1248457
12821624,12864023,DistanceLtd,28606.65625,1248457


In [29]:
diagnostics_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12821626 entries, 0 to 12821625
Data columns (total 4 columns):
 #   Column   Dtype 
---  ------   ----- 
 0   Id       int64 
 1   Name     object
 2   Value    object
 3   FaultId  int64 
dtypes: int64(2), object(2)
memory usage: 391.3+ MB


In [30]:
# To get the on-board diagnostics at the time of the fault code, we can match the **RecordID** to the **FaultId**.
diagnostics_df.loc[diagnostics_df['FaultId'] == 1]

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1
5,6,AcceleratorPedal,0,1
6,7,IntakeManifoldTemperature,78.8,1
7,8,FuelRate,0,1
8,9,FuelLtd,12300.907429328,1
9,10,EngineRpm,0,1


## Merge FAULTS and DIAGNOSTICS

### Diagnostics 1-to-1 with faults

In [ ]:
# Pivot DIAGNOSTICS wider
diagnostics_pivot_wider_df = diagnostics_df.pivot(
    columns='Name',
    index='FaultId',
    values='Value'
)
diagnostics_pivot_wider_df.sample(5)

In [ ]:
# Replace commas with decimal points and convert to floats
float_columns = [
    'AcceleratorPedal',
    'BarometricPressure',
    'DistanceLtd',
    'EngineCoolantTemperature',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'EngineTimeLtd',
    'FuelLevel',
    'FuelLtd',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'SwitchedBatteryVoltage',
    'Throttle',
    'TurboBoostPressure'
]

for col in float_columns:
    print(col)
    diagnostics_pivot_wider_df[col] = diagnostics_pivot_wider_df[col].str.replace(pat=',', repl='.').astype(float)

In [15]:
faults_diagnostics_df = pd.merge(
    left=faults_df,
    right=diagnostics_pivot_wider_df,
    how='inner',
    left_on='RecordID',
    right_on='FaultId',
    validate='1:1'
)
faults_diagnostics_df['Derate_Target'].isna().sum()

NameError: name 'diagnostics_pivot_wider_df' is not defined

In [18]:
faults_diagnostics_df.to_csv('../data/faults_diagnostics.csv', index=False)

In [19]:
# Split data into training and testing
cutoff_date = '2018-12-31 23:59:59'

training_faults_diagnostics_df = faults_diagnostics_df[faults_diagnostics_df['EventTimeStamp'] <= cutoff_date].reset_index()
testing_faults_diagnostics_df = faults_diagnostics_df[faults_diagnostics_df['EventTimeStamp'] > cutoff_date].reset_index()

In [20]:
training_faults_diagnostics_df.shape

(1058069, 50)

In [21]:
testing_faults_diagnostics_df.shape

(129266, 50)

In [22]:
training_faults_diagnostics_df.to_csv('../data/training_faults_diagnostics.csv', index=False)
testing_faults_diagnostics_df.to_csv('../data/testing_faults_diagnostics.csv', index=False)

In [23]:
training_faults_diagnostics_df['IsFullDerate'].sum()

np.int64(444)

In [34]:
training_faults_diagnostics_df[training_faults_diagnostics_df['Derate_Target'] != 0].groupby(['EquipmentID', 'EventTimeStamp'])[['Derate_Target', 'IsFullDerate']].sum().head(20)

Derate_Target  IsFullDerate
EquipmentID EventTimeStamp                                  
105349576   2018-07-06 03:53:05              3             0
            2018-07-06 03:53:06              1             0
            2018-07-06 04:53:05              1             0
            2018-07-06 09:42:48              2             1
105427203   2018-04-27 06:07:55              2             1
1329        2015-02-25 13:53:08              8             1
1339        2015-06-12 07:41:27              1             0
            2015-06-12 07:59:36              1             0
            2015-06-12 08:24:15              1             0
            2015-06-12 15:35:22              2             1
1366        2015-06-12 03:57:39              1             0
            2015-06-12 03:57:49              1             0
            2015-06-12 06:13:27              2             1
            2015-06-12 20:49:02              1             0
            2015-06-12 22:51:02              1             0
            2015-06-12 23:30:09              1             0
            2015-06-13 02:31:08              2             0
            2015-06-13 03:33:30              2             1
            2015-07-03 15:10:45              1             0
            2015-07-03 19:10:35              2             1